
# Exercício Prático 2 — Identificação e Correção de **Deadlock** em Python (**multiprocessing**)

Este notebook guia, passo a passo, a **reprodução**, o **diagnóstico** e a **correção** de um *deadlock* usando **processos** (`multiprocessing`),
evitando que o **GIL** do CPython masque o comportamento concorrente (como pode ocorrer em `threading`).

> **Objetivo didático:** observar um deadlock real entre dois fluxos de execução competindo por **dois locks** adquiridos em ordem inversa,
relacionar com as **condições de Coffman**, construir o **grafo de espera/alocação**, e aplicar duas estratégias de correção:
1) **Ordenação global** de aquisição de locks  
2) **Try-lock com timeout** (evitando espera circular)

---

## Nota sobre execução em notebooks (Windows/macOS)
Em alguns ambientes, `multiprocessing` usa o método **spawn**, que pode exigir:

```python
if __name__ == "__main__":
    ...
```

Em Jupyter geralmente funciona, mas se houver travamentos/loops estranhos, rode como script `.py`
ou configure explicitamente:

```python
import multiprocessing as mp
mp.set_start_method("spawn", force=True)  # ou "fork" em Linux
```



## 1) Cenário: dois locks, ordem inversa (potencial deadlock)

Estrutura típica do problema:

- Processo P1: pega `lock1` e depois tenta `lock2`
- Processo P2: pega `lock2` e depois tenta `lock1`

Se P1 segurar `lock1` enquanto P2 segura `lock2`, ambos ficam esperando indefinidamente → **deadlock**.


In [1]:

import multiprocessing as mp
import time


In [2]:

def p1(lock1, lock2, started_evt):
    started_evt.set()
    with lock1:
        time.sleep(0.2)  # aumenta chance de interleaving / colisão
        with lock2:
            pass

def p2(lock1, lock2, started_evt):
    started_evt.set()
    with lock2:
        time.sleep(0.2)
        with lock1:
            pass



## 2) Tarefa 1 — Executar e observar possível bloqueio

⚠️ Para não “congelar” o notebook, detectaremos deadlock usando `join(timeout=...)`.

Interpretação:
- Se após um tempo limite os processos ainda estiverem vivos, é forte evidência de deadlock.


In [3]:

def run_deadlock_demo(timeout_s=2.0):
    lock1 = mp.Lock()
    lock2 = mp.Lock()

    e1 = mp.Event()
    e2 = mp.Event()

    proc1 = mp.Process(target=p1, args=(lock1, lock2, e1))
    proc2 = mp.Process(target=p2, args=(lock1, lock2, e2))

    proc1.start(); proc2.start()

    # aguarda ambos iniciarem (melhora reprodutibilidade)
    e1.wait(1.0); e2.wait(1.0)

    proc1.join(timeout_s)
    proc2.join(timeout_s)

    deadlock_suspected = proc1.is_alive() and proc2.is_alive()

    if deadlock_suspected:
        # encerra processos para não deixar “pendurado” no ambiente
        proc1.terminate()
        proc2.terminate()
        proc1.join()
        proc2.join()

    return deadlock_suspected

for i in range(5):
    print(i, "deadlock?", run_deadlock_demo(timeout_s=1.5))


0 deadlock? True
1 deadlock? True
2 deadlock? True
3 deadlock? True
4 deadlock? True



## 3) Condições de Coffman (identificação formal no código)

Quando o deadlock ocorre, as quatro condições estão presentes:

1. **Exclusão mútua**: locks são exclusivos (um processo por vez).
2. **Hold and wait (posse e espera)**: cada processo segura um lock e espera o outro.
3. **Não preempção**: não existe “tomada forçada” do lock.
4. **Espera circular**: P1 espera lock2 (com P2) e P2 espera lock1 (com P1).

> Tarefa: aponte exatamente em que parte do código cada condição se manifesta.



## 4) Tarefa 2 — Análise estrutural: Grafo de espera (Wait-For Graph)

Como cada lock tem **uma instância**, podemos usar o grafo de espera (WFG):

- Nó = processo
- Aresta `Pi → Pj` significa: **Pi espera recurso mantido por Pj**

No deadlock:
- P1 segura `lock1` e espera `lock2` (com P2) → **P1 → P2**
- P2 segura `lock2` e espera `lock1` (com P1) → **P2 → P1**

Ciclo:
- **P1 → P2 → P1**

✅ Ciclo no WFG ⇒ deadlock (instância única).



## 5) Tarefa 3 — Correção (Estratégia A): Ordenação global de locks

A correção clássica impõe uma **ordem global** para aquisição de recursos.

Regra:
> Sempre adquirir `lock1` antes de `lock2`.

Isso remove a **espera circular**.


In [4]:

def p1_ordered(lock1, lock2):
    with lock1:
        time.sleep(0.2)
        with lock2:
            pass

def p2_ordered(lock1, lock2):
    # mesma ordem (lock1 → lock2)
    with lock1:
        time.sleep(0.2)
        with lock2:
            pass

def run_ordering_fix(timeout_s=2.0):
    lock1 = mp.Lock()
    lock2 = mp.Lock()

    proc1 = mp.Process(target=p1_ordered, args=(lock1, lock2))
    proc2 = mp.Process(target=p2_ordered, args=(lock1, lock2))

    proc1.start(); proc2.start()
    proc1.join(timeout_s); proc2.join(timeout_s)

    ok = (not proc1.is_alive()) and (not proc2.is_alive())

    if not ok:
        proc1.terminate(); proc2.terminate()
        proc1.join(); proc2.join()
    return ok

for i in range(5):
    print(i, "completed?", run_ordering_fix(timeout_s=2.0))


0 completed? True
1 completed? True
2 completed? True
3 completed? True
4 completed? True



## 6) Correção (Estratégia B): Try-lock com timeout

Outra abordagem é evitar espera indefinida:

- tenta adquirir o segundo lock com `timeout`
- se falhar, libera o primeiro e tenta novamente (*backoff*)

Isso impede espera circular permanente.


In [5]:

def worker_trylock(lock_first, lock_second, rounds=300, timeout=0.01):
    for _ in range(rounds):
        acquired1 = lock_first.acquire(timeout=timeout)
        if not acquired1:
            continue
        try:
            acquired2 = lock_second.acquire(timeout=timeout)
            if acquired2:
                try:
                    # seção crítica
                    time.sleep(0.001)
                    return True
                finally:
                    lock_second.release()
        finally:
            lock_first.release()
        time.sleep(0.001)  # backoff simples
    return False

def run_trylock_fix(timeout_s=3.0):
    lock1 = mp.Lock()
    lock2 = mp.Lock()

    # simula "ordem inversa" (cada um tenta começar por um lock diferente)
    pA = mp.Process(target=worker_trylock, args=(lock1, lock2))
    pB = mp.Process(target=worker_trylock, args=(lock2, lock1))

    pA.start(); pB.start()
    pA.join(timeout_s); pB.join(timeout_s)

    ok = (not pA.is_alive()) and (not pB.is_alive())

    if not ok:
        pA.terminate(); pB.terminate()
        pA.join(); pB.join()
    return ok

for i in range(5):
    print(i, "completed?", run_trylock_fix(timeout_s=3.0))


0 completed? True
1 completed? True
2 completed? True
3 completed? True
4 completed? True



## 7) O que entregar (Checklist)

1) Evidência de deadlock no código base (prints + join timeout)  
2) Identificação das 4 condições de Coffman no código  
3) Grafo de espera (WFG) com ciclo `P1 → P2 → P1`  
4) Correção por ordenação global (código + evidência)  
5) Correção por try-lock timeout (código + evidência)  
6) Discussão de trade-offs (corretude vs desempenho vs engenharia)
